# 01 — TSMOM Baseline

## Block 01 — Data Audit

### Objetivo
Verificar que el universo diario Gold está completo, consistente y alineado antes de calcular retornos o cualquier objeto de estrategia.

### Alcance
Universo: `SPY`, `QQQ`, `XLE`, `GLD`. Ventana solicitada: `2018-01-02` hasta la última sesión común disponible no posterior a `2026-11-17`.

### Fuente y causalidad
Este notebook lee exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. No se permite look-ahead: las etapas posteriores deberán utilizar únicamente información disponible en cada instante de decisión.


In [1]:
from datetime import date
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

SYMBOLS = ("SPY", "QQQ", "XLE", "GLD")
WINDOW_START = date(2018, 1, 2)
WINDOW_END = date(2026, 11, 17)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
GOLD_ROOT


WindowsPath('C:/Users/avazq/OneDrive/Escritorio/cienciadatos/portafolio/quant-research-notes/Momentum_systematic_strategies/data/gold/daily_series/etf')

In [2]:
REQUIRED_COLUMNS = (
    "session_date", "open", "high", "low", "close", "adjusted_close",
    "volume", "dividend_amount", "split_coefficient",
)

frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
    missing_columns = set(REQUIRED_COLUMNS).difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{symbol} no contiene columnas requeridas: {sorted(missing_columns)}")
    frames[symbol] = frame.filter(
        pl.col("session_date").is_between(WINDOW_START, WINDOW_END, closed="both")
    ).sort("session_date")

{symbol: frame.shape for symbol, frame in frames.items()}


{'SPY': (2232, 14), 'QQQ': (2232, 14), 'XLE': (2232, 14), 'GLD': (2232, 14)}

In [3]:
def ohlc_violation_count(frame: pl.DataFrame) -> int:
    invalid_bar = (
        (pl.col("open") <= 0) | (pl.col("high") <= 0) |
        (pl.col("low") <= 0) | (pl.col("close") <= 0) |
        (pl.col("high") < pl.col("low")) |
        (pl.col("high") < pl.col("open")) |
        (pl.col("high") < pl.col("close")) |
        (pl.col("low") > pl.col("open")) |
        (pl.col("low") > pl.col("close"))
    )
    return frame.select(invalid_bar.sum()).item()

audit_rows = []
for symbol, frame in frames.items():
    null_counts = frame.select(pl.all().null_count()).to_dicts()[0]
    audit_rows.append({
        "symbol": symbol,
        "sessions": frame.height,
        "start": frame.get_column("session_date").min(),
        "end": frame.get_column("session_date").max(),
        "nulls_total": sum(null_counts.values()),
        "duplicate_dates": frame.select(pl.col("session_date").is_duplicated().sum()).item(),
        "ohlc_violations": ohlc_violation_count(frame),
        "adjusted_close_available": "adjusted_close" in frame.columns,
        "positive_dividend_sessions": frame.select((pl.col("dividend_amount") > 0).sum()).item(),
        "non_unit_split_sessions": frame.select((pl.col("split_coefficient") != 1).sum()).item(),
    })

asset_audit = pl.DataFrame(audit_rows)
asset_audit


symbol,sessions,start,end,nulls_total,duplicate_dates,ohlc_violations,adjusted_close_available,positive_dividend_sessions,non_unit_split_sessions
str,i64,date,date,i64,i64,i64,bool,i64,i64
"""SPY""",2232,2018-01-02,2026-11-17,0,0,0,true,35,0
"""QQQ""",2232,2018-01-02,2026-11-17,0,0,0,true,36,0
"""XLE""",2232,2018-01-02,2026-11-17,0,0,0,true,36,1
"""GLD""",2232,2018-01-02,2026-11-17,0,0,0,true,0,0


In [4]:
panel = None
for symbol in SYMBOLS:
    series = frames[symbol].select(
        "session_date", pl.col("adjusted_close").alias(symbol)
    )
    panel = series if panel is None else panel.join(series, on="session_date", how="inner")

panel = panel.sort("session_date")
panel_nulls = panel.select(pl.all().null_count()).to_dicts()[0]
panel_summary = {
    "rows": panel.height,
    "columns": panel.width,
    "start": panel.get_column("session_date").min(),
    "end": panel.get_column("session_date").max(),
    "nulls_total": sum(panel_nulls.values()),
    "duplicate_dates": panel.select(pl.col("session_date").is_duplicated().sum()).item(),
}

assert panel.columns == ["session_date", *SYMBOLS]
assert panel_summary["start"] == WINDOW_START
assert panel_summary["end"] <= WINDOW_END
assert panel_summary["nulls_total"] == 0

print(panel_summary)
panel.head()


{'rows': 2232, 'columns': 5, 'start': datetime.date(2018, 1, 2), 'end': datetime.date(2026, 11, 17), 'nulls_total': 0, 'duplicate_dates': 0}


session_date,SPY,QQQ,XLE,GLD
date,f64,f64,f64,f64
2018-01-02,235.20385,149.695763,25.380449,125.15
2018-01-03,236.69154,151.150312,25.760551,124.82
2018-01-04,237.689168,151.414775,25.916047,125.46
2018-01-05,239.273121,152.93544,25.905681,125.33
2018-01-08,239.710677,153.530483,26.061177,125.31


In [6]:
normalized_panel = panel.with_columns([
    (pl.col(symbol) / pl.col(symbol).first()).alias(symbol)
    for symbol in SYMBOLS
])

figure = go.Figure()
for symbol in SYMBOLS:
    figure.add_scatter(
        x=normalized_panel.get_column("session_date").to_list(),
        y=normalized_panel.get_column(symbol).to_list(),
        mode="lines",
        name=symbol,
    )

figure.update_layout(
    title="Normalized adjusted close (P_t / P_0)",
    xaxis_title="Session date",
    yaxis_title="P_t / P_0",
    template="plotly_white",
    legend_title="ETF",
)
figure.show()


### Resultados de la ejecución — 2026-11-17

| Activo | Sesiones | Rango | Nulos | Fechas duplicadas | Violaciones OHLC | `adjusted_close` | Dividendos positivos | Splits no unitarios |
| --- | ---: | --- | ---: | ---: | ---: | --- | ---: | ---: |
| SPY | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 35 | 0 |
| QQQ | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 36 | 0 |
| XLE | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 36 | 1 |
| GLD | 2,232 | 2018-01-02 → 2026-11-17 | 0 | 0 | 0 | Disponible | 0 | 0 |

La ventana común exacta es `2018-01-02 → 2026-11-17`. El panel wide `session_date | SPY | QQQ | XLE | GLD` tiene dimensión **2,232 × 5**, sin nulos ni fechas duplicadas. El split no unitario de XLE queda registrado como dato corporativo informativo; no constituye una violación OHLC.

**PASS — ready for Block 02.**

## Block 02 — Returns
_Pendiente. No implementado en esta sesión._

## Block 03 — TSMOM Signals
_Pendiente. No implementado en esta sesión._

## Block 04 — Causal PnL
_Pendiente. No implementado en esta sesión._

## Block 05 — Diagnostics
_Pendiente. No implementado en esta sesión._
